# §2.5.3 — KL을 표본으로 추정하면 무슨 일이 생기는가

> 딥러닝 교재 · 1부 2장 5절 3항 (🐍)
> 선행: §2.5.2(깁스 부등식과 접선 부등식) · §2.5.3(가우시안 KL의 폐형식)

## 이 노트북이 답하는 질문

1. **몬테카를로 KL 추정량이 폐형식과 맞는가?** §2.5.3의 결과를 참값으로 쓴다.
2. **어느 추정량을 써야 하는가?** 세 가지를 편향과 분산으로 비교한다.
3. **발산 추정치가 음수가 나올 수 있는가?** — 나온다. 그리고 그것이 무엇을 뜻하는가.
4. **널리 권장되는 추정량이 언제 무너지는가?** 답은 **$q$ 의 꼬리가 $p$ 보다 두꺼울 때**이고, 그때 분산이 **무한**이다.

**예상 실행 시간** CPU 단일 코어 약 45초 (`FAST = True`이면 약 15초).

이 노트북은 §37.2의 예고이기도 하다. 여기서는 **두 밀도를 모두 아는데도** 추정이 어렵다.
밀도를 모르고 표본만 있는 상황(상호정보량 추정)은 훨씬 나쁘다.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False
SEED     = 20260809
N_SAMP   = 100        # 한 번의 추정에 쓰는 표본 수
N_REP    = 200_000    # 추정을 반복하는 횟수 (추정량의 분포를 보기 위해)
SAVE_PDF = False
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────
if FAST:
    N_REP = 40_000

CB = ['#000000','#E69F00','#56B4E9','#009E73','#D55E00','#0072B2','#CC79A7','#F0E442']
plt.rcParams.update({'figure.dpi':120,'font.size':10,'axes.grid':True,'grid.alpha':0.3,
                     'axes.prop_cycle':plt.cycler(color=CB),'figure.autolayout':True})
import matplotlib.font_manager as fm
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic','Malgun Gothic','AppleGothic','Noto Sans CJK KR',
                            'Noto Sans KR','NanumBarunGothic','Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en
_fi=[0]
def show(name):
    _fi[0] += 1
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        plt.savefig(os.path.join(FIG_DIR, f'fig_2_5_4_{_fi[0]}_{name}.pdf'),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()

print(f"numpy {np.__version__} | N_REP={N_REP:,} | 한글폰트: {KO_FONT or '없음(영문 라벨)'}")

---
## 1. 세 추정량

$p$ 에서 표본 $z_1,\dots,z_n$ 을 뽑고, 두 밀도를 모두 계산할 수 있다고 하자.
비율을 $r(z) \triangleq q(z)/p(z)$ 라 쓰면 세 가지 추정량이 있다.

| | 식 | 성질 |
|---|---|---|
| $k_1$ | $-\log r$ | 정의 그대로. **불편** |
| $k_2$ | $\tfrac12(\log r)^2$ | 항상 $\ge 0$. **편향** |
| $k_3$ | $r - 1 - \log r$ | **불편이면서 항상 $\ge 0$** |

**$k_1$ 이 불편인 것은 정의에서 자명하다.**

$$\mathbb{E}_{z\sim p}\big[-\log r(z)\big] = \mathbb{E}_p\left[\log\frac{p}{q}\right] = \mathrm{KL}(p\|q)$$

**$k_3$ 도 불편이다.** $\mathbb{E}_p[r] = \int p\cdot\frac{q}{p} = \int q = 1$ 이므로

$$\mathbb{E}_p[k_3] = \underbrace{\mathbb{E}_p[r]}_{=1} - 1 - \mathbb{E}_p[\log r] = \mathrm{KL}(p\|q)$$

**그리고 $k_3 \ge 0$ 이 점별로 성립한다** — §2.5.2 4절의 접선 부등식 $\log t \le t-1$ 이 바로 그것이다.
$k_3$ 는 그 부등식의 **간극**을 그대로 쓴 것이다.

> **한 표본으로도 음수가 나오지 않는다는 것이 $k_3$ 의 매력입니다.** $k_1$ 은 그렇지 않습니다.

In [ ]:
def kl_gauss(m1, s1, m2, s2):
    # §2.5.3의 폐형식
    return np.log(s2/s1) + (s1**2 + (m1-m2)**2)/(2*s2**2) - 0.5

def logpdf(z, m, s):
    return -0.5*np.log(2*np.pi) - np.log(s) - (z-m)**2/(2*s**2)

def estimators(z, m1, s1, m2, s2):
    logr = logpdf(z, m2, s2) - logpdf(z, m1, s1)      # log r = log q - log p
    r = np.exp(logr)
    return {'k1': -logr, 'k2': 0.5*logr**2, 'k3': r - 1.0 - logr}

def repeat(m1, s1, m2, s2, n, R, seed, budget=2_000_000):
    # (R, n) 을 한꺼번에 만들면 메모리가 크므로 원소 수 예산에 맞춰 나눈다
    g = np.random.default_rng(seed); out = {k: [] for k in ('k1','k2','k3')}
    chunk = max(500, budget // max(n, 1))
    done = 0
    while done < R:
        c = min(chunk, R - done)
        z = g.normal(m1, s1, (c, n))
        e = estimators(z, m1, s1, m2, s2)
        for k in out:
            out[k].append(e[k].mean(axis=1))
        done += c
    return {k: np.concatenate(v) for k, v in out.items()}

P = (0.0, 1.0)           # p = N(0,1)
Q = (1.0, 1.0)           # q = N(1,1)
TRUE = kl_gauss(*P, *Q)
print(f"p = N({P[0]},{P[1]}²),  q = N({Q[0]},{Q[1]}²)")
print(f"폐형식 KL(p‖q) = {TRUE:.6f}   (§2.5.3)")

---
## 2. 편향과 분산

표본 수 $n$ 을 바꿔 가며 세 추정량의 분포를 본다. **참값은 폐형식으로 알고 있다.**

In [ ]:
NS = [10, 30, 100, 300, 1000, 3000]
tab = {k: {'mean': [], 'std': []} for k in ('k1','k2','k3')}
neg_frac = []
for n in NS:
    e = repeat(*P, *Q, n, N_REP//4, SEED + n)
    for k in tab:
        tab[k]['mean'].append(e[k].mean()); tab[k]['std'].append(e[k].std())
    neg_frac.append(float((e['k1'] < 0).mean()))

print(f"참 KL = {TRUE:.4f}")
print("     n        k1 평균 ± 표준편차       k2 평균 ± 표준편차       k3 평균 ± 표준편차    k1<0 비율")
for i, n in enumerate(NS):
    print(f"{n:>6}   {tab['k1']['mean'][i]:.4f} ± {tab['k1']['std'][i]:.4f}   "
          f"{tab['k2']['mean'][i]:.4f} ± {tab['k2']['std'][i]:.4f}   "
          f"{tab['k3']['mean'][i]:.4f} ± {tab['k3']['std'][i]:.4f}    {neg_frac[i]:.4f}")
print(f"\n-> k1, k3 는 평균이 참값과 일치한다 (불편).")
print(f"   k2 는 {tab['k2']['mean'][-1]/TRUE:.2f}배로 치우쳐 있다 — 표본을 늘려도 사라지지 않는다.")
print(f"   같은 n에서 k3 의 표준편차가 k1 보다 작다.")

In [ ]:
n_show = 30
e = repeat(*P, *Q, n_show, N_REP//4, SEED+7)
fig, axes = plt.subplots(1, 2, figsize=(9.8, 3.6))

bins = np.linspace(-0.4, 1.6, 80)
for k, c, nm in [('k1', CB[4], '$k_1 = -\\log r$'),
                 ('k2', CB[1], '$k_2 = \\frac{1}{2}(\\log r)^2$'),
                 ('k3', CB[3], '$k_3 = r - 1 - \\log r$')]:
    axes[0].hist(e[k], bins=bins, density=True, histtype='step', lw=1.8, color=c, label=nm)
axes[0].axvline(TRUE, color=CB[0], lw=2, label=lab('참값 (폐형식)','truth'))
axes[0].axvline(0, color='0.6', lw=1.0, ls=':')
axes[0].set_xlabel(lab('추정값','estimate')); axes[0].set_yticks([])
axes[0].set_title(lab(f'추정량의 분포 ($n$ = {n_show})', f'estimator distributions'), fontsize=10)
axes[0].legend(fontsize=8)

for k, c in [('k1', CB[4]), ('k3', CB[3])]:
    axes[1].loglog(NS, tab[k]['std'], 'o-', ms=4, color=c, label=k)
axes[1].loglog(NS, tab['k1']['std'][0]*np.sqrt(NS[0]/np.array(NS)), 'k:', lw=1.2,
               label=lab(r'$n^{-1/2}$', r'$n^{-1/2}$'))
axes[1].set_xlabel(lab('표본 수 $n$','samples $n$'))
axes[1].set_ylabel(lab('추정량의 표준편차','std of estimator'))
axes[1].set_title(lab('둘 다 $n^{-1/2}$ 로 줄지만 상수가 다르다',
                      'both decay as $n^{-1/2}$'), fontsize=10)
axes[1].legend(fontsize=8)
show('estimators')

> **왼쪽 그림에서 $k_1$ 의 분포가 0 왼쪽으로 넘어갑니다.** 발산의 추정치가 음수인 것이며,
> §2.5.2가 증명한 $\mathrm{KL} \ge 0$ 과 모순되는 값입니다.
>
> 물론 추정량이 불편이라고 **개별 추정치가 참값의 성질을 물려받지는 않습니다.**
> 그러나 실무에서 이것이 문제가 됩니다 — KL을 벌점으로 쓰면서 음수가 나오면 최적화가 이상해집니다.
> $k_3$ 는 점별로 비음이라 그 문제가 없습니다.

---
## 3. 그러나 $k_3$ 가 항상 낫지는 않다

여기서 이야기가 바뀐다. $p$ 와 $q$ 를 점점 멀리 떼어 놓으며 두 추정량을 비교한다.

In [ ]:
CASES = [
    (lab('평균 이동 d=0.5','mean shift 0.5'),   (0.,1.), (0.5,1.)),
    (lab('평균 이동 d=2','mean shift 2'),       (0.,1.), (2.,1.)),
    (lab('평균 이동 d=5','mean shift 5'),       (0.,1.), (5.,1.)),
    (lab('q가 좁다 σ2=0.3','q narrow 0.3'),     (0.,1.), (0.,0.3)),
    (lab('q가 좁다 σ2=0.1','q narrow 0.1'),     (0.,1.), (0.,0.1)),
    (lab('q가 넓다 σ2=3','q wide 3'),           (0.,1.), (0.,3.)),
]
print("  설정                      참 KL     k1 상대오차   k3 상대오차   비")
rows = []
for nm, pp, qq in CASES:
    t = kl_gauss(*pp, *qq)
    e = repeat(*pp, *qq, N_SAMP, N_REP//4, SEED+11)
    r1 = e['k1'].std()/t; r3 = e['k3'].std()/t
    rows.append((nm, t, r1, r3))
    print(f"  {nm:<24} {t:8.4f}   {r1:10.4f}   {r3:10.4f}   {r3/r1:7.2f}")
print("\n-> 평균 이동이 크거나 q가 넓으면 k3 가 오히려 크게 나쁘다.")

### 왜 그런가 — $k_3$ 의 분산은 비율의 분산이다

$k_3 = r - 1 - \log r$ 에서 큰 $r$ 이 나오면 첫 항이 지배한다. 그러므로

$$\mathrm{Var}_p[k_3] \ \approx \ \mathrm{Var}_p[r] \ = \ \int \frac{q^2}{p} - 1$$

**이것은 카이제곱 발산이고, KL보다 훨씬 빨리 자란다.** 그리고 **유한하지 않을 수도 있다.**

가우시안에서는 조건이 명시적으로 나온다.

$$\int \frac{q^2}{p} < \infty \iff \frac{2}{\sigma_2^2} - \frac{1}{\sigma_1^2} > 0 \iff \sigma_2^2 < 2\sigma_1^2$$

> **$q$ 의 꼬리가 $p$ 보다 두꺼우면 $\mathrm{Var}[k_3] = \infty$ 입니다.** 중심극한정리가 적용되지 않고,
> 표본을 늘려도 $n^{-1/2}$ 로 줄지 않습니다.

In [ ]:
def chi2_minus_one(m1, s1, m2, s2):
    a = 2/s2**2 - 1/s1**2
    if a <= 0:
        return np.inf
    val = (s1/(s2**2*np.sqrt(a)))*np.exp(-(2*m2**2/s2**2 - m1**2/s1**2)/2
                                         + (2*m2/s2**2 - m1/s1**2)**2/(2*a))
    return val - 1.0

print("  설정                      참 KL      Var_p[r]        분산")
for nm, pp, qq in CASES:
    v = chi2_minus_one(*pp, *qq)
    print(f"  {nm:<24} {kl_gauss(*pp,*qq):8.4f}   {v:13.4g}   "
          f"{'유한' if np.isfinite(v) else '무한'}")
print("\n-> 평균 이동 d=5 는 유한하지만 7.2e10 이다. 유한과 실용은 다르다.")

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.0))
xs = np.arange(len(rows))
ax.bar(xs - 0.2, [r[2] for r in rows], 0.4, color=CB[4], label='$k_1$')
ax.bar(xs + 0.2, [r[3] for r in rows], 0.4, color=CB[3], label='$k_3$')
ax.set_yscale('log')
ax.set_xticks(xs); ax.set_xticklabels([r[0] for r in rows], rotation=30, ha='right', fontsize=8)
ax.set_ylabel(lab(f'상대 오차 (표준편차 / 참값), $n$={N_SAMP}',
                  f'relative error, $n$={N_SAMP}'))
ax.set_title(lab('$k_3$ 는 두 분포가 가까울 때만 유리하다',
                 '$k_3$ wins only when $p$ and $q$ are close'), fontsize=10)
ax.legend(fontsize=9)
show('k1_vs_k3')

---
## 4. 무한 분산을 어떻게 알아채는가

무한 분산은 **조용히 지나갑니다.** 표본을 조금만 보면 아무 문제가 없어 보입니다.
진단법은 **시행 수를 늘려 가며 경험 표준편차가 안정되는지 보는 것**입니다.

In [ ]:
def growth(pp, qq, seed, Rs):
    g = np.random.default_rng(seed); acc = {'k1': [], 'k3': []}
    out = []
    for R in Rs:
        while sum(len(a) for a in acc['k1']) < R:
            z = g.normal(pp[0], pp[1], (max(500, 2_000_000//N_SAMP), N_SAMP))
            e = estimators(z, *pp, *qq)
            acc['k1'].append(e['k1'].mean(1)); acc['k3'].append(e['k3'].mean(1))
        A = np.concatenate(acc['k1'])[:R]; B = np.concatenate(acc['k3'])[:R]
        out.append((R, A.std(), B.std(), B.max()))
    return out

RS = [1000, 5000, 20_000, 100_000] + ([] if FAST else [500_000])
fig, axes = plt.subplots(1, 2, figsize=(9.8, 3.6), sharey=False)
for ax, (nm, qq, tag) in zip(axes, [
        (lab('q가 넓다 (σ2=3)','q wide'), (0.,3.), lab('무한 분산','infinite var')),
        (lab('q가 좁다 (σ2=0.3)','q narrow'), (0.,0.3), lab('유한 분산','finite var'))]):
    res = growth(P, qq, SEED+31, RS)
    R_ = [r[0] for r in res]
    ax.semilogx(R_, [r[1] for r in res], 'o-', ms=4, color=CB[4], label='$k_1$')
    ax.semilogx(R_, [r[2] for r in res], 's-', ms=4, color=CB[3], label='$k_3$')
    ax.set_xlabel(lab('시행 수','number of replications'))
    ax.set_title(f'{nm} — {tag}', fontsize=10)
    ax.legend(fontsize=8)
    print(f"\n{nm}  (참 KL = {kl_gauss(*P,*qq):.4f})")
    print("     시행 수    k1 표준편차   k3 표준편차   k3 최댓값")
    for R, s1_, s3_, mx in res:
        print(f"   {R:>9,}   {s1_:11.4f}   {s3_:11.4f}   {mx:11.2f}")
axes[0].set_ylabel(lab('경험 표준편차','empirical std'))
fig.suptitle(lab('왼쪽에서 $k_3$ 의 표준편차가 멈추지 않는다 — 무한 분산의 징후',
                 'the left panel never stabilizes'), y=1.04, fontsize=10)
show('infinite_variance')

> ### 이것이 이 노트북의 핵심 관찰이다
>
> 왼쪽에서 $k_1$ 의 표준편차는 시행 수와 무관하게 일정한데 **$k_3$ 는 계속 자랍니다.**
> 최댓값도 함께 자라며 상한이 보이지 않습니다.
>
> **경험 표준편차가 안정되지 않으면 그 추정량의 분산은 유한하지 않을 가능성이 큽니다.**
> 그리고 이 진단은 참값을 몰라도 할 수 있습니다 — 시행을 늘려 보기만 하면 됩니다.
>
> $k_3$ 는 KL 벌점을 쓰는 여러 실무 상황에서 권장되는 추정량인데, **그 상황들이 대개
> $p \approx q$ 이기 때문에** 잘 작동합니다. 두 분포가 벌어지는 순간 이 성질이 사라집니다.

---
## 5. 실무 지침과 §37.2 예고

| 상황 | 권장 |
|---|---|
| $p \approx q$ (KL이 작다) | $k_3$ — 불편이고 비음이며 분산이 작다 |
| 두 분포가 멀다 | $k_1$ — 분산이 유한하고 안정적 |
| **판정 방법** | 시행을 늘려 경험 표준편차가 안정되는지 확인 |
| 어느 경우든 | **폐형식이 있으면 그것을 쓴다** (§2.5.3) |

> ⚠︎ **이 노트북은 가장 쉬운 경우였습니다.** 두 밀도를 **모두 정확히 알고** 있었고,
> 분포가 가우시안이라 참값도 폐형식으로 알았습니다. 그런데도 추정량 선택이 문제가 됐습니다.
>
> **밀도를 모르고 표본만 있으면 훨씬 나쁩니다.** 상호정보량 추정이 그 상황이며,
> $I(X;Y) = \mathrm{KL}\big(p(x,y) \,\|\, p(x)p(y)\big)$ 이므로 결국 KL 추정 문제입니다.
> §37.2가 그 어려움을 본격적으로 다룹니다.

---
## 6. 자기 점검

1. $k_2$ 의 편향이 표본을 늘려도 사라지지 않았다. **왜인가?** (힌트: $k_2$ 의 기댓값은 무엇인가)
2. 2절에서 $k_1$ 이 음수를 냈다. **$n$ 을 늘리면 이 문제가 사라지는가?** 표에서 확인하라.
3. 3절에서 "$q$ 가 넓다"가 문제였고 "$q$ 가 좁다"는 괜찮았다. **비대칭의 이유**를 $r = q/p$ 로 설명하라.
4. $\mathrm{KL}(q\|p)$ 를 $q$ 의 표본으로 추정하면 3절의 결론이 어떻게 뒤집히는가?

In [ ]:
# 자기 점검 4의 확인 — 방향을 뒤집으면
print("KL(p‖q) 를 p 표본으로  vs  KL(q‖p) 를 q 표본으로   (σ2 = 3)")
qq = (0.0, 3.0)
for (a, bq, nm) in [(P, qq, "KL(p‖q), p에서 표집"), (qq, P, "KL(q‖p), q에서 표집")]:
    t = kl_gauss(*a, *bq)
    e = repeat(*a, *bq, N_SAMP, N_REP//4, SEED+41)
    v = chi2_minus_one(*a, *bq)
    print(f"  {nm:<22} 참값 {t:7.4f}   k1 std {e['k1'].std():7.4f}   "
          f"k3 std {e['k3'].std():9.4f}   Var[r] {'무한' if not np.isfinite(v) else f'{v:.3g}'}")
print("\n-> 방향을 뒤집으면 넓은 쪽에서 표집하게 되어 비율이 유계가 되고, k3 가 다시 안정된다.")
print("   '어느 방향의 KL인가'가 추정 난이도까지 바꾼다 (§2.5.5).")

---
## 7. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `N_SAMP` | 0절 | 100 | 한 추정에 쓰는 표본 수 |
| `N_REP` | 0절 | 200,000 | 추정량의 분포를 보는 시행 수. **4절의 진단은 이 값에 의존한다** |
| `Q` | 1절 | $(1, 1)$ | 비교할 분포 |
| `CASES` | 3절 | 여섯 설정 | $p$ 와 $q$ 의 조합 |
| `RS` | 4절 | 1e3 ~ 5e5 | 진단용 시행 수 격자 |

**권하는 첫 실험** — 3절의 `CASES` 에 $\sigma_2 = 1.5$ 를 추가하십시오. $\sigma_2^2 = 2.25 > 2\sigma_1^2 = 2$ 이므로
**분산이 무한한 경계 바로 바깥**입니다. $\sigma_2 = 1.4$ ($\sigma_2^2 = 1.96 < 2$) 와 나란히 놓으면
**조건이 만족되는지 여부가 거동을 완전히 갈라놓는 것**을 볼 수 있습니다.

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")